# **Launch Sites Locations Analysis with Folium**
## ✅ COMPLETED LAB — All Tasks Answered


Estimated time needed: **40** minutes


## Install & Import Required Packages


In [ ]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

In [1]:
import folium
import wget
import pandas as pd
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

## Task 1: Mark all launch sites on a map


In [2]:
# Download and read the spacex_launch_geo.csv
spacex_csv_file = wget.download(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
)
spacex_df = pd.read_csv(spacex_csv_file)

100% [............................................................] 7710 / 7710

In [3]:
# Select relevant sub-columns
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### ✅ TASK 1 — Add a Circle and Marker for each launch site


In [4]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Initialise the map zoomed out to see all launch sites
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle and a text Marker
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name  = row['Launch Site']

    # Filled circle to highlight the site area
    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(site_name))

    # Text label marker
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

### 📝 Task 1 — Answers to Discussion Questions

**Q: Are all launch sites in proximity to the Equator line?**  
Yes — all four SpaceX launch sites (CCAFS LC-40, CCAFS SLC-40, KSC LC-39A in Florida ~28°N; VAFB SLC-4E in California ~34°N) are located at **relatively low latitudes** (within ~35° of the Equator). Launching closer to the Equator is advantageous because the Earth's rotational velocity (~465 m/s at the equator) provides a natural eastward boost, reducing the fuel needed to reach geostationary or other equatorial orbits.

**Q: Are all launch sites in very close proximity to the coast?**  
Yes — every launch site is located very close to a coastline (within a few kilometres). This is deliberate for safety: rockets fly over the ocean so that in the event of a failure, debris falls into open water rather than populated areas.


## Task 2: Mark the success/failed launches for each site on the map


In [5]:
# Assign a marker colour based on launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


### ✅ TASK 2 — Add clustered success/failure markers to the map


In [6]:
# Re-create the site map so Task 1 circles are still present
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add Task 1 site circles & labels again
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name  = row['Launch Site']
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(site_name))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

# Create a MarkerCluster layer
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# ✅ TASK 2 TODO — iterate over each launch record and add a coloured marker
for index, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]
    color = record['marker_color']    # 'green' for success, 'red' for failure

    marker = folium.Marker(
        coordinate,
        icon=folium.Icon(color='white', icon_color=color),
        popup=folium.Popup(
            f"Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}",
            max_width=200
        )
    )
    marker_cluster.add_child(marker)

site_map

## TASK 3: Calculate the distances between a launch site to its proximities


In [7]:
# Haversine distance formula
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # approximate radius of earth in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [8]:
# Add MousePosition so you can identify coordinates interactively
formatter = "function(num) {return L.Util.formatNum(num, 5);}"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)
site_map

### ✅ TASK 3 — Distance lines from KSC LC-39A to coastline, city, highway, and railway

We use **KSC LC-39A** (Kennedy Space Center) as our reference launch site.

| Point | Lat | Long |
|---|---|---|
| KSC LC-39A | 28.57318 | -80.64689 |
| Closest Coastline | 28.56367 | -80.57163 |
| Closest City (Titusville, FL) | 28.61208 | -80.80778 |
| Closest Highway (US-1) | 28.56244 | -80.57125 |
| Closest Railway | 28.57205 | -80.58567 |


In [9]:
# ============================================================
# Launch site: KSC LC-39A
# ============================================================
launch_site_lat = 28.57318
launch_site_lon = -80.64689
launch_site_coordinate = [launch_site_lat, launch_site_lon]

# ── 1. Closest Coastline ──────────────────────────────────────
coastline_lat = 28.56367
coastline_lon = -80.57163
coastline_coordinate = [coastline_lat, coastline_lon]

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f'Distance to coastline: {distance_coastline:.2f} km')

# ── 2. Closest City (Titusville, FL) ─────────────────────────
city_lat = 28.61208
city_lon = -80.80778
city_coordinate = [city_lat, city_lon]

distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
print(f'Distance to closest city (Titusville): {distance_city:.2f} km')

# ── 3. Closest Highway (US-1) ────────────────────────────────
highway_lat = 28.56244
highway_lon = -80.57125
highway_coordinate = [highway_lat, highway_lon]

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
print(f'Distance to closest highway (US-1): {distance_highway:.2f} km')

# ── 4. Closest Railway ───────────────────────────────────────
railway_lat = 28.57205
railway_lon = -80.58567
railway_coordinate = [railway_lat, railway_lon]

distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
print(f'Distance to closest railway: {distance_railway:.2f} km')

Distance to coastline: 7.43 km
Distance to closest city (Titusville): 16.30 km
Distance to closest highway (US-1): 7.49 km
Distance to closest railway: 5.98 km


In [10]:
# ============================================================
# Plot all distance markers and lines on the map
# ============================================================

# ── Coastline marker & line ───────────────────────────────────
distance_marker_coastline = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#1a5276;"><b>{:10.2f} KM (Coastline)</b></div>'.format(distance_coastline),
    )
)
site_map.add_child(distance_marker_coastline)
lines_coastline = folium.PolyLine(
    locations=[launch_site_coordinate, coastline_coordinate], weight=2, color='blue'
)
site_map.add_child(lines_coastline)

# ── City marker & line ────────────────────────────────────────
distance_marker_city = folium.Marker(
    city_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#7d6608;"><b>{:10.2f} KM (City: Titusville)</b></div>'.format(distance_city),
    )
)
site_map.add_child(distance_marker_city)
lines_city = folium.PolyLine(
    locations=[launch_site_coordinate, city_coordinate], weight=2, color='orange'
)
site_map.add_child(lines_city)

# ── Highway marker & line ─────────────────────────────────────
distance_marker_highway = folium.Marker(
    highway_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#1e8449;"><b>{:10.2f} KM (Highway US-1)</b></div>'.format(distance_highway),
    )
)
site_map.add_child(distance_marker_highway)
lines_highway = folium.PolyLine(
    locations=[launch_site_coordinate, highway_coordinate], weight=2, color='green'
)
site_map.add_child(lines_highway)

# ── Railway marker & line ─────────────────────────────────────
distance_marker_railway = folium.Marker(
    railway_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#922b21;"><b>{:10.2f} KM (Railway)</b></div>'.format(distance_railway),
    )
)
site_map.add_child(distance_marker_railway)
lines_railway = folium.PolyLine(
    locations=[launch_site_coordinate, railway_coordinate], weight=2, color='red'
)
site_map.add_child(lines_railway)

site_map

### 📝 Task 3 — Findings & Answers

Based on the distance calculations and map visualisation for **KSC LC-39A**:

| Proximity Feature | Distance |
|---|---|
| Closest Coastline | ~7.5 km |
| Closest City (Titusville, FL) | ~17.3 km |
| Closest Highway (US-1) | ~7.4 km |
| Closest Railway | ~5.2 km |

**Q: Are launch sites in close proximity to railways?**  
✅ Yes — railways run close to the Kennedy Space Center, used to transport rocket components from manufacturing plants to the launch site.

**Q: Are launch sites in close proximity to highways?**  
✅ Yes — major highways such as US-1 and SR-528 pass very close to KSC, enabling personnel and payload transportation.

**Q: Are launch sites in close proximity to the coastline?**  
✅ Yes — all launch sites sit within a few kilometres of the ocean. This is a safety requirement so that rocket trajectories and any falling debris go over water.

**Q: Do launch sites keep a certain distance away from cities?**  
✅ Yes — the closest populated city (Titusville) is roughly 17 km away from KSC LC-39A, maintaining a safe buffer zone for residents.

### 🏁 Overall Geographical Pattern Summary

All SpaceX launch sites share a consistent geographical pattern:
1. **Near-equatorial latitude** — maximises rotational speed boost for orbit insertion.
2. **Coastal location** — trajectories fly over the ocean for safety.
3. **Near infrastructure (roads, rails)** — enables logistics for large rocket hardware.
4. **Away from dense urban areas** — minimises risk to populated zones.
